In [ ]:
import time
import copy
import pickle

import numpy as np
import pandas as pd
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

from sklearn.model_selection import cross_val_score
from sklearn.metrics import mean_squared_error, r2_score, make_scorer

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor

from xgboost import XGBRegressor

from hyperopt import fmin, tpe, hp, STATUS_OK, Trials
from hyperopt.pyll import scope

In [ ]:
def display_df(df):
    display(HTML(df.to_html()))
    return None

In [ ]:
def check_smiles_in(simles, smiles_lst):
    assert isinstance(simles, str)
    assert isinstance(smiles_lst, list)
    return simles.lower().strip() in [s.lower().strip() for s in smiles_lst]

In [ ]:
def rmse(estimator, X_eval, y_eval):
    y_hat = estimator.predict(X_eval)
    return np.sqrt(mean_squared_error(y_eval, y_hat))


def r2(estimator, X_eval, y_eval):
    y_hat = estimator.predict(X_eval)
    return r2_score(y_eval, y_hat)


def peason_r(estimator, X_eval, y_eval):
    y_hat = estimator.predict(X_eval)
    return np.corrcoef(y_eval, y_hat)[0, 1]


def peason_r_metric(y_true, y_pred):
    return np.corrcoef(y_true, y_pred)[0, 1]

peason_r_score = make_scorer(peason_r_metric)

In [ ]:
def whole_to_int(a_dict):
    new_dict = copy.deepcopy(a_dict)
    for k, v in new_dict.items():
        if np.isclose(np.round(v), v):
            new_dict[k] = int(new_dict[k])
    return new_dict


def hyperopt_reg(regressor,
                 params_tuned, 
                 X_train, y_train,
                 num_eval,
                 params_fixed=None,
                 rstate=None):
    
    time_start = time.time()
    if params_fixed is None:
        params_fixed = {}
    
    def objective(params):
        regressor.set_params(**params_fixed, **params)
        # may use scoring='r2', "neg_mean_squared_error"
        neg_mse = cross_val_score(regressor, X_train, y_train, cv=10, scoring="neg_mean_squared_error").mean()
        #r2 = cross_val_score(regressor, X_train, y_train, cv=10, scoring="r2").mean()
        #pearson_r = cross_val_score(regressor, X_train, y_train, cv=10, scoring=peason_r_score).mean()
        return {"loss": -neg_mse, "status": STATUS_OK}
    
    if rstate is not None:
        rstate = np.random.RandomState(rstate)
        
    trials = Trials()
    best_params = fmin(objective, 
                      params_tuned, 
                      algo=tpe.suggest, 
                      max_evals=num_eval, 
                      trials=trials,
                      rstate=rstate)
    
    best_params = whole_to_int(best_params)
    best_model = regressor.set_params(**params_fixed, **best_params)
    best_model.fit(X_train, y_train)
    
    time_end = time.time()
    time_elapse = time_end - time_start
    print("Time elapsed: %0.5f s" % time_elapse)
    return trials, best_params, best_model

## Load train/test

In [ ]:
pdXY = pd.read_csv("data/process/pdXY_ab42_train_test_rdkit_descriptors_109ft_imputed_std.csv")

SMILES_TRAIN_TEST = pdXY["smiles"].unique().tolist()

PDY_COLS = ["new_id", "smiles", "dG", "code", "train_test", "smiles_len"]
PDX_COLS = sorted([col for col in pdXY.columns if col not in PDY_COLS])
print("PDX_COLS", len(PDX_COLS))

print(pdXY.shape)
display_df(pdXY.head())


X_train = pdXY.loc[pdXY["train_test"] == "train", PDX_COLS].copy().values
y_train = pdXY.loc[pdXY["train_test"] == "train", "dG"].copy().values
print(X_train.shape, y_train.shape)

X_test = pdXY.loc[pdXY["train_test"] == "test", PDX_COLS].copy().values
y_test = pdXY.loc[pdXY["train_test"] == "test", "dG"].copy().values
print(X_test.shape, y_test.shape)

## XGBOOST

In [ ]:
assert True

params = {
    "max_depth": scope.int(hp.quniform("max_depth", 2, 8, 1)),
    "min_child_weight": scope.int(hp.quniform("min_child_weight", 1, 10, 1)), 
    "subsample": hp.uniform("subsample", 0.4, 1.0),
    "colsample_bytree": hp.uniform("colsample_bytree", 0.4, 1.0),
    "reg_lambda": hp.loguniform("reg_lambda", np.log(0.00001), np.log(100)),
    #"reg_alpha": hp.loguniform("reg_alpha", np.log(0.001), np.log(1000)),
    "learning_rate": hp.loguniform("learning_rate", np.log(0.0001), np.log(1.)),
    #"gamma": hp.uniform("gamma", 0., 5.),
}

params_fixed = {
    "tree_method": "gpu_hist",
    "predictor": "gpu_predictor",
    "n_estimators": 300
}


num_eval = 100

xgb = XGBRegressor()

trials, best_params, best_model = hyperopt_reg(xgb, params, X_train, y_train, num_eval, params_fixed=params_fixed)
print("best_params:", best_params)

print("Train RMSE:", rmse(best_model, X_train, y_train))
print("Train Pearson's R:", peason_r(best_model, X_train, y_train))

print("Test RMSE:", rmse(best_model, X_test, y_test))
print("Test Pearson's R:", peason_r(best_model, X_test, y_test))

pickle.dump(best_model, open("models/xgb/xbg_01.pkl", "wb"))

### predict for train test

In [ ]:
assert False

pdXY = pd.read_csv("data/process/pdXY_ab42_train_test_rdkit_descriptors_109ft_imputed_std.csv")

print(pdXY.shape)
display_df(pdXY.head())

model = pickle.load(open("models/xgb/xbg_01.pkl", "rb"))
pdXY["pred"] = model.predict(pdXY[PDX_COLS].values)

df00 = pdXY[PDY_COLS+["pred"]]
print(df00.shape)
display_df(df00.head())

df00.to_csv("results/xgb/train_test_pred.csv", index=False)

## predict for chembl33

In [ ]:
assert False


pdXY = pd.read_csv("data/process/pdXY_chembl33_rdkit_descriptors_109ft_imputed_std.csv")
print(pdXY.shape)
display_df(pdXY.head(1))

# remove inf rows
for cname in PDX_COLS:
    pdXY[cname] = pdXY[cname].apply(lambda n: np.nan if n in [-np.inf, np.inf] else n)
pdXY = pdXY.dropna(subset=PDX_COLS)
print("pdXY", pdXY.shape)
display_df(pdXY.head(1))


model = pickle.load(open("models/xgb/xbg_01.pkl", "rb"))
pdXY["pred"] = model.predict(pdXY[PDX_COLS].values)

df00 = pdXY[PDY_COLS+["pred"]]
print(df00.shape)
display_df(df00.head(1))

df00.to_csv("results/xgb/chembl33_pred.csv", index=False)

In [ ]:
df00.sort_values(by="pred").head(1000)

## predict for chembl36excl33

In [ ]:
assert False


pdXY = pd.read_csv("data/process/pdXY_chembl36excl33_rdkit_descriptors_109ft_imputed_std.csv")
print(pdXY.shape)
display_df(pdXY.head(1))

# remove inf rows
for cname in PDX_COLS:
    pdXY[cname] = pdXY[cname].apply(lambda n: np.nan if n in [-np.inf, np.inf] else n)
pdXY = pdXY.dropna(subset=PDX_COLS)
print("pdXY", pdXY.shape)
display_df(pdXY.head(1))


model = pickle.load(open("models/xgb/xbg_01.pkl", "rb"))
pdXY["pred"] = model.predict(pdXY[PDX_COLS].values)

df00 = pdXY[PDY_COLS+["pred"]]
print(df00.shape)
display_df(df00.head(1))

df00.to_csv("results/xgb/chembl36excl33_pred.csv", index=False)

In [ ]:
df00.sort_values(by="pred").head(100)

## predict for amylovis201

In [ ]:
assert False


pdXY = pd.read_csv("data/process/pdXY_amylovis201_rdkit_descriptors_109ft_imputed_std.csv")
print(pdXY.shape)
display_df(pdXY.head(1))

# remove inf rows
for cname in PDX_COLS:
    pdXY[cname] = pdXY[cname].apply(lambda n: np.nan if n in [-np.inf, np.inf] else n)
pdXY = pdXY.dropna(subset=PDX_COLS)
print("pdXY", pdXY.shape)
display_df(pdXY.head(1))


model = pickle.load(open("models/xgb/xbg_01.pkl", "rb"))
pdXY["pred"] = model.predict(pdXY[PDX_COLS].values)

df00 = pdXY[PDY_COLS+["pred"]]
print(df00.shape)
display_df(df00.head(1))

df00.to_csv("results/xgb/amylovis201_pred.csv", index=False)

## predict for ref01

In [ ]:
assert True


pdXY = pd.read_csv("data/process/pdXY_ref01_rdkit_descriptors_109ft_imputed_std.csv")
print(pdXY.shape)
display_df(pdXY.head(1))

# remove inf rows
for cname in PDX_COLS:
    pdXY[cname] = pdXY[cname].apply(lambda n: np.nan if n in [-np.inf, np.inf] else n)
pdXY = pdXY.dropna(subset=PDX_COLS)
print("pdXY", pdXY.shape)
display_df(pdXY.head(1))


model = pickle.load(open("models/xgb/xbg_01.pkl", "rb"))
pdXY["pred"] = model.predict(pdXY[PDX_COLS].values)

df00 = pdXY[PDY_COLS+["pred"]]
print(df00.shape)
display_df(df00.head(1))

df00.to_csv("results/xgb/ref01_pred.csv", index=False)